# 04 - Train Random Forest

Train a baseline Random Forest classifier for comparison with the production LightGBM model.

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path('..') / '..'
DATA_PATH = ROOT / 'data' / 'raw' / 'Student_Performance.csv'
MODEL_DIR = ROOT / 'models'
METRICS_DIR = ROOT / 'reports' / 'metrics'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(DATA_PATH)

In [ ]:
target = df['final_grade'].astype(str).str.lower()
features = df.drop(columns=['student_id', 'overall_score', 'final_grade'])
categorical_columns = features.select_dtypes(include=['object']).columns.tolist()
numeric_columns = [col for col in features.columns if col not in categorical_columns]

train_x, test_x, train_y, test_y = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42,
    stratify=target,
)

preprocessor = ColumnTransformer(
    transformers=[
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
        ('numeric', 'passthrough', numeric_columns),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)),
    ]
)
model.fit(train_x, train_y)
predictions = model.predict(test_x)
accuracy = accuracy_score(test_y, predictions)
accuracy

In [ ]:
report = classification_report(test_y, predictions, output_dict=True, zero_division=0)
joblib.dump(model, MODEL_DIR / 'student_random_forest_model.joblib')
(METRICS_DIR / 'random_forest_metrics.json').write_text(
    json.dumps({'accuracy': accuracy, 'classification_report': report}, indent=2),
    encoding='utf-8',
)
pd.DataFrame(report).T